# 03 - Training a Small Transformer for Text Classification

---

## Learning Objectives

By the end of this notebook, you will be able to:

- Build a **small Transformer encoder** for text classification from scratch
- Implement the full pipeline: **tokenize, numericalize, pad, DataLoader, train**
- Construct the architecture: **Embedding -> Positional Encoding -> N Encoder Blocks -> Mean Pool -> FC**
- Train the model on a text classification task (sklearn 20newsgroups subset)
- Compare Transformer performance with an **LSTM baseline**
- Plot **training curves** and a **confusion matrix**

## Prerequisites

- Completed Notebook 01 (self-attention) and Notebook 02 (Transformer architecture)
- Comfortable with PyTorch `nn.Module`, `DataLoader`, training loops
- Basic understanding of text preprocessing (tokenization, vocabulary building)

## Table of Contents

1. [Data: 20 Newsgroups Subset](#1-data-20-newsgroups-subset)
2. [Text Preprocessing Pipeline](#2-text-preprocessing-pipeline)
3. [Dataset and DataLoader](#3-dataset-and-dataloader)
4. [Transformer Encoder for Classification](#4-transformer-encoder-for-classification)
5. [Training Pipeline](#5-training-pipeline)
6. [Code: Train the Transformer](#6-code-train-the-transformer)
7. [Code: Plot Training Curves](#7-code-plot-training-curves)
8. [Code: Confusion Matrix](#8-code-confusion-matrix)
9. [LSTM Baseline Comparison](#9-lstm-baseline-comparison)
10. [Exercise: Tune Hyperparameters](#10-exercise-tune-hyperparameters)
11. [Common Mistakes & Debugging Tips](#11-common-mistakes--debugging-tips)

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt
import math
import re
from collections import Counter

from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

set_global_seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

---
## 1. Data: 20 Newsgroups Subset

We use a **4-category subset** of the 20 Newsgroups dataset to keep training fast:

- `sci.space` -- space and astronomy
- `rec.sport.baseball` -- baseball
- `comp.graphics` -- computer graphics
- `talk.politics.mideast` -- Middle East politics

These are sufficiently different that even a small model can learn to distinguish them.

In [ ]:
# Load 4-category subset
categories = ['sci.space', 'rec.sport.baseball', 'comp.graphics', 'talk.politics.mideast']

newsgroups = fetch_20newsgroups(
    subset='all',
    categories=categories,
    remove=('headers', 'footers', 'quotes'),  # remove metadata for cleaner text
    random_state=42
)

texts = newsgroups.data
labels = newsgroups.target
label_names = newsgroups.target_names

print(f"Total samples: {len(texts)}")
print(f"Classes: {label_names}")
print(f"Label distribution: {dict(zip(label_names, np.bincount(labels)))}")
print(f"\nSample text (first 200 chars):")
print(texts[0][:200])

---
## 2. Text Preprocessing Pipeline

We build a simple tokenizer and vocabulary from scratch:

1. **Tokenize:** Lowercase, remove non-alphanumeric, split on whitespace
2. **Build vocabulary:** Map each unique token to an integer index
3. **Numericalize:** Convert token lists to integer lists
4. **Pad/Truncate:** Ensure all sequences have the same length

In [ ]:
def simple_tokenizer(text):
    """Lowercase, remove non-alphanumeric chars, split on whitespace."""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    tokens = text.split()
    return tokens


class Vocabulary:
    """Simple vocabulary: token <-> index mapping."""
    def __init__(self, max_size=10000):
        self.token2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2token = {0: '<PAD>', 1: '<UNK>'}
        self.max_size = max_size
    
    def build(self, tokenized_texts):
        """Build vocabulary from list of token lists."""
        counter = Counter()
        for tokens in tokenized_texts:
            counter.update(tokens)
        
        # Keep most common tokens (reserve 2 for PAD and UNK)
        for token, _ in counter.most_common(self.max_size - 2):
            idx = len(self.token2idx)
            self.token2idx[token] = idx
            self.idx2token[idx] = token
    
    def numericalize(self, tokens):
        """Convert tokens to indices."""
        return [self.token2idx.get(t, self.token2idx['<UNK>']) for t in tokens]
    
    def __len__(self):
        return len(self.token2idx)


# Tokenize all texts
tokenized_texts = [simple_tokenizer(text) for text in texts]

# Build vocabulary
vocab = Vocabulary(max_size=10000)
vocab.build(tokenized_texts)

print(f"Vocabulary size: {len(vocab)}")
print(f"Sample tokenization: {tokenized_texts[0][:10]}")
print(f"Sample numericalization: {vocab.numericalize(tokenized_texts[0][:10])}")

In [ ]:
# Numericalize all texts and inspect length distribution
numericalized = [vocab.numericalize(tokens) for tokens in tokenized_texts]
lengths = [len(seq) for seq in numericalized]

print(f"Sequence length stats:")
print(f"  Min: {min(lengths)}, Max: {max(lengths)}, Mean: {np.mean(lengths):.0f}, Median: {np.median(lengths):.0f}")

plt.hist(lengths, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Sequence Length (tokens)')
plt.ylabel('Count')
plt.title('Distribution of Document Lengths')
plt.axvline(x=200, color='red', linestyle='--', label='max_len=200')
plt.legend()
plt.tight_layout()
plt.show()

# We will truncate/pad to max_len=200
MAX_LEN = 200
print(f"\nUsing MAX_LEN={MAX_LEN} (covers {sum(1 for l in lengths if l <= MAX_LEN)/len(lengths)*100:.1f}% of documents fully)")

---
## 3. Dataset and DataLoader

We create a PyTorch `Dataset` that handles padding/truncation and returns `(input_ids, label)` pairs.

In [ ]:
class TextClassificationDataset(Dataset):
    """Dataset for text classification with padding/truncation."""
    
    def __init__(self, sequences, labels, max_len=200, pad_idx=0):
        self.sequences = sequences
        self.labels = labels
        self.max_len = max_len
        self.pad_idx = pad_idx
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        seq = self.sequences[idx][:self.max_len]  # truncate
        # Pad to max_len
        padded = seq + [self.pad_idx] * (self.max_len - len(seq))
        return torch.tensor(padded, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)


# Train/test split
set_global_seed(42)
train_seqs, test_seqs, train_labels, test_labels = train_test_split(
    numericalized, labels, test_size=0.2, random_state=42, stratify=labels
)

train_dataset = TextClassificationDataset(train_seqs, train_labels, max_len=MAX_LEN)
test_dataset = TextClassificationDataset(test_seqs, test_labels, max_len=MAX_LEN)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train size: {len(train_dataset)}, Test size: {len(test_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")

# Verify a batch
sample_batch = next(iter(train_loader))
print(f"Batch input shape: {sample_batch[0].shape}")
print(f"Batch label shape: {sample_batch[1].shape}")

---
## 4. Transformer Encoder for Classification

Our architecture:

```
Token IDs (batch, seq_len)
       |
  Embedding (vocab_size, d_model)
       |
  Positional Encoding
       |
  N x Encoder Blocks
       |
  Mean Pooling (over sequence dim, ignoring padding)
       |
  Fully Connected -> num_classes
```

**Mean pooling** averages token representations to get a single document vector. We mask out padding tokens so they do not contribute to the mean.

In [ ]:
# Building blocks (from Notebook 02)

def scaled_dot_product_attention(Q, K, V, mask=None):
    """Scaled dot-product attention."""
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(~mask, float('-inf'))
    attn_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V)
    return output, attn_weights


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, Q, K, V, mask=None):
        batch_size = Q.size(0)
        Q = self.W_Q(Q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(K).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(V).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask=mask)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.W_O(attn_output), attn_weights


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        attn_output, _ = self.self_attn(x, x, x, mask=mask)
        x = self.norm1(x + self.dropout1(attn_output))
        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_output))
        return x


print("Building blocks loaded.")

In [ ]:
class TransformerClassifier(nn.Module):
    """
    Transformer Encoder for Text Classification.
    
    Architecture:
        Embedding -> Positional Encoding -> N Encoder Blocks -> Mean Pool -> FC
    """
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers,
                 num_classes, max_len=512, dropout=0.1, pad_idx=0):
        super().__init__()
        self.d_model = d_model
        self.pad_idx = pad_idx
        
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        
        self.encoder_layers = nn.ModuleList([
            EncoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        
        self.norm = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, num_classes)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, input_ids):
        """
        Args:
            input_ids: (batch, seq_len) -- token IDs
        Returns:
            logits: (batch, num_classes)
        """
        # Create padding mask: (batch, 1, 1, seq_len) for broadcasting
        pad_mask = (input_ids != self.pad_idx).unsqueeze(1).unsqueeze(2)  # True = real token
        
        # Embed and scale
        x = self.embedding(input_ids) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        
        # Encoder blocks
        for layer in self.encoder_layers:
            x = layer(x, mask=pad_mask)
        
        x = self.norm(x)  # (batch, seq_len, d_model)
        
        # Mean pooling (exclude padding)
        # mask shape: (batch, seq_len)
        mask = (input_ids != self.pad_idx).float()  # (batch, seq_len)
        x = x * mask.unsqueeze(-1)  # zero out padding positions
        x = x.sum(dim=1) / mask.sum(dim=1, keepdim=True).clamp(min=1)  # (batch, d_model)
        
        # Classify
        x = self.dropout(x)
        logits = self.fc(x)  # (batch, num_classes)
        return logits


# Test the classifier
set_global_seed(42)
model = TransformerClassifier(
    vocab_size=len(vocab),
    d_model=64,
    n_heads=4,
    d_ff=128,
    n_layers=2,
    num_classes=len(label_names),
    max_len=MAX_LEN,
    dropout=0.1,
    pad_idx=0
)

# Forward pass test
sample_input = sample_batch[0]
logits = model(sample_input)
print(f"Input shape:  {sample_input.shape}")
print(f"Output shape: {logits.shape}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## 5. Training Pipeline

Standard training loop with:

- **Loss:** `CrossEntropyLoss`
- **Optimizer:** Adam with learning rate scheduling
- **Metrics:** Accuracy tracked per epoch on train and test sets

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    """Train for one epoch, return average loss and accuracy."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for input_ids, labels in dataloader:
        input_ids, labels = input_ids.to(device), labels.to(device)
        
        optimizer.zero_grad()
        logits = model(input_ids)
        loss = criterion(logits, labels)
        loss.backward()
        
        # Gradient clipping to prevent explosion
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item() * input_ids.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / total, correct / total


def evaluate(model, dataloader, criterion, device):
    """Evaluate on a dataset, return average loss and accuracy."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for input_ids, labels in dataloader:
            input_ids, labels = input_ids.to(device), labels.to(device)
            logits = model(input_ids)
            loss = criterion(logits, labels)
            
            total_loss += loss.item() * input_ids.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    return total_loss / total, correct / total


print("Training functions defined.")

---
## 6. Code: Train the Transformer

In [ ]:
set_global_seed(42)

# Hyperparameters
D_MODEL = 64
N_HEADS = 4
D_FF = 128
N_LAYERS = 2
NUM_EPOCHS = 15
LR = 1e-3

# Create model
transformer_model = TransformerClassifier(
    vocab_size=len(vocab),
    d_model=D_MODEL,
    n_heads=N_HEADS,
    d_ff=D_FF,
    n_layers=N_LAYERS,
    num_classes=len(label_names),
    max_len=MAX_LEN,
    dropout=0.1,
    pad_idx=0
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(transformer_model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print(f"Model parameters: {sum(p.numel() for p in transformer_model.parameters()):,}")
print(f"Training for {NUM_EPOCHS} epochs...")
print()

# Training history
history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(
        transformer_model, train_loader, optimizer, criterion, device
    )
    test_loss, test_acc = evaluate(
        transformer_model, test_loader, criterion, device
    )
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['test_loss'].append(test_loss)
    history['train_acc'].append(train_acc)
    history['test_acc'].append(test_acc)
    
    if (epoch + 1) % 3 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Test Loss: {test_loss:.4f} Acc: {test_acc:.4f}")

print(f"\nFinal Test Accuracy: {history['test_acc'][-1]:.4f}")

---
## 7. Code: Plot Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, NUM_EPOCHS + 1)

# Loss
axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss', markersize=4)
axes[0].plot(epochs_range, history['test_loss'], 'r-o', label='Test Loss', markersize=4)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Test Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, history['train_acc'], 'b-o', label='Train Accuracy', markersize=4)
axes[1].plot(epochs_range, history['test_acc'], 'r-o', label='Test Accuracy', markersize=4)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training & Test Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0, 1])

plt.suptitle('Transformer Classifier Training Curves', fontsize=14)
plt.tight_layout()
plt.show()

---
## 8. Code: Confusion Matrix

In [ ]:
def get_predictions(model, dataloader, device):
    """Get all predictions and true labels from a dataloader."""
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for input_ids, labels in dataloader:
            input_ids = input_ids.to(device)
            logits = model(input_ids)
            preds = logits.argmax(dim=1).cpu()
            all_preds.append(preds)
            all_labels.append(labels)
    return torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()


preds, true_labels = get_predictions(transformer_model, test_loader, device)

# Classification report
print("Classification Report:")
print(classification_report(true_labels, preds, target_names=label_names))

# Confusion matrix
cm = confusion_matrix(true_labels, preds)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.set_title('Confusion Matrix - Transformer Classifier')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Label ticks
short_names = [name.split('.')[-1] for name in label_names]
ax.set_xticks(range(len(short_names)))
ax.set_xticklabels(short_names, rotation=45, ha='right')
ax.set_yticks(range(len(short_names)))
ax.set_yticklabels(short_names)

# Annotate cells
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color=color, fontsize=12)

ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.tight_layout()
plt.show()

---
## 9. LSTM Baseline Comparison

For perspective, we train a simple **bidirectional LSTM** classifier with a comparable parameter count.

Architecture: **Embedding -> BiLSTM -> Last Hidden State -> FC**

In [ ]:
class LSTMClassifier(nn.Module):
    """Bidirectional LSTM for text classification."""
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 n_layers=1, dropout=0.1, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=n_layers,
            batch_first=True, bidirectional=True, dropout=dropout if n_layers > 1 else 0
        )
        self.fc = nn.Linear(hidden_dim * 2, num_classes)  # *2 for bidirectional
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, input_ids):
        embedded = self.dropout(self.embedding(input_ids))  # (batch, seq, embed)
        output, (hidden, _) = self.lstm(embedded)
        # Concatenate final hidden states from both directions
        # hidden: (2*n_layers, batch, hidden_dim)
        hidden_cat = torch.cat([hidden[-2], hidden[-1]], dim=1)  # (batch, hidden*2)
        logits = self.fc(self.dropout(hidden_cat))
        return logits


# Train LSTM baseline
set_global_seed(42)

lstm_model = LSTMClassifier(
    vocab_size=len(vocab),
    embed_dim=64,
    hidden_dim=64,
    num_classes=len(label_names),
    n_layers=1,
    dropout=0.1,
    pad_idx=0
).to(device)

lstm_params = sum(p.numel() for p in lstm_model.parameters())
transformer_params = sum(p.numel() for p in transformer_model.parameters())
print(f"LSTM parameters:        {lstm_params:,}")
print(f"Transformer parameters: {transformer_params:,}")
print()

lstm_criterion = nn.CrossEntropyLoss()
lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=1e-3)
lstm_scheduler = optim.lr_scheduler.StepLR(lstm_optimizer, step_size=5, gamma=0.5)

lstm_history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}

print(f"Training LSTM for {NUM_EPOCHS} epochs...")
for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(
        lstm_model, train_loader, lstm_optimizer, lstm_criterion, device
    )
    test_loss, test_acc = evaluate(
        lstm_model, test_loader, lstm_criterion, device
    )
    lstm_scheduler.step()
    
    lstm_history['train_loss'].append(train_loss)
    lstm_history['test_loss'].append(test_loss)
    lstm_history['train_acc'].append(train_acc)
    lstm_history['test_acc'].append(test_acc)
    
    if (epoch + 1) % 3 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Test Loss: {test_loss:.4f} Acc: {test_acc:.4f}")

print(f"\nFinal LSTM Test Accuracy: {lstm_history['test_acc'][-1]:.4f}")

In [ ]:
# Compare training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, NUM_EPOCHS + 1)

# Loss comparison
axes[0].plot(epochs_range, history['test_loss'], 'b-o', label='Transformer', markersize=4)
axes[0].plot(epochs_range, lstm_history['test_loss'], 'r-s', label='LSTM', markersize=4)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Test Loss')
axes[0].set_title('Test Loss Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy comparison
axes[1].plot(epochs_range, history['test_acc'], 'b-o', label='Transformer', markersize=4)
axes[1].plot(epochs_range, lstm_history['test_acc'], 'r-s', label='LSTM', markersize=4)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('Test Accuracy Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0, 1])

plt.suptitle('Transformer vs LSTM Baseline', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Final Test Accuracy -- Transformer: {history['test_acc'][-1]:.4f}, LSTM: {lstm_history['test_acc'][-1]:.4f}")

---
## 10. Exercise: Tune Hyperparameters

**Task:** Experiment with the Transformer architecture by varying:

1. **`n_heads`**: Try 2 vs 4 vs 8 (keep `d_model` divisible by `n_heads`)
2. **`n_layers`**: Try 1 vs 2 vs 4
3. **`d_model`**: Try 32 vs 64 vs 128

Record test accuracy for each configuration. Which combination works best?

```python
# Your code here
# Suggested structure:
#
# configs = [
#     {'d_model': 64, 'n_heads': 2, 'n_layers': 2},
#     {'d_model': 64, 'n_heads': 4, 'n_layers': 2},
#     {'d_model': 64, 'n_heads': 8, 'n_layers': 2},
#     {'d_model': 64, 'n_heads': 4, 'n_layers': 1},
#     {'d_model': 64, 'n_heads': 4, 'n_layers': 4},
#     {'d_model': 32, 'n_heads': 4, 'n_layers': 2},
#     {'d_model': 128, 'n_heads': 4, 'n_layers': 2},
# ]
#
# for config in configs:
#     set_global_seed(42)
#     model = TransformerClassifier(vocab_size=len(vocab), num_classes=len(label_names),
#                                   d_ff=config['d_model']*2, max_len=MAX_LEN, **config).to(device)
#     # Train for a few epochs and record test accuracy
#     ...
```

In [ ]:
# Try the exercise yourself before looking at the solution!





### Solution

In [ ]:
# ----- Solution -----
configs = [
    {'d_model': 64, 'n_heads': 2, 'n_layers': 2, 'label': 'h=2, L=2, d=64'},
    {'d_model': 64, 'n_heads': 4, 'n_layers': 2, 'label': 'h=4, L=2, d=64'},
    {'d_model': 64, 'n_heads': 8, 'n_layers': 2, 'label': 'h=8, L=2, d=64'},
    {'d_model': 64, 'n_heads': 4, 'n_layers': 1, 'label': 'h=4, L=1, d=64'},
    {'d_model': 64, 'n_heads': 4, 'n_layers': 4, 'label': 'h=4, L=4, d=64'},
]

TUNE_EPOCHS = 8  # fewer epochs for speed
results = []

for config in configs:
    set_global_seed(42)
    label = config.pop('label')
    model = TransformerClassifier(
        vocab_size=len(vocab),
        d_ff=config['d_model'] * 2,
        num_classes=len(label_names),
        max_len=MAX_LEN,
        dropout=0.1,
        pad_idx=0,
        **config
    ).to(device)
    
    opt = optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    
    for ep in range(TUNE_EPOCHS):
        train_one_epoch(model, train_loader, opt, crit, device)
    
    _, test_acc = evaluate(model, test_loader, crit, device)
    n_params = sum(p.numel() for p in model.parameters())
    results.append((label, test_acc, n_params))
    print(f"{label:20s} | Test Acc: {test_acc:.4f} | Params: {n_params:,}")
    config['label'] = label  # restore

print("\nBest config:", max(results, key=lambda x: x[1])[0])

---
## 11. Common Mistakes & Debugging Tips

**1. Forgetting to mask padding in mean pooling**
- If padding tokens (value 0) contribute to the mean, the representation is diluted
- Always zero out padding positions before averaging and divide by the actual sequence length

**2. Not scaling embeddings by $\sqrt{d_{\text{model}}}$**
- Positional encoding has fixed magnitude; without scaling, embeddings may dominate or be dominated by PE
- The scaling keeps both signals at a comparable magnitude

**3. Using too large a model for small datasets**
- Transformers are data-hungry; a 6-layer model on 3000 samples will overfit quickly
- Start small (1-2 layers, small `d_model`) and scale up with data

**4. Learning rate too high**
- Transformers are sensitive to learning rate; values above 1e-3 often cause divergence
- Use warmup if training a larger model, and always use gradient clipping

**5. Padding mask shape mismatch**
- Attention scores are `(batch, n_heads, seq, seq)`
- Padding mask must broadcast correctly: `(batch, 1, 1, seq)` masks key positions

**6. Comparing LSTM and Transformer unfairly**
- Ensure similar parameter counts for a fair comparison
- Transformers excel with larger data and longer sequences; LSTMs may win on tiny datasets

**7. Not shuffling training data**
- If data is sorted by class, the model sees one class at a time, leading to poor convergence
- Always use `shuffle=True` in the training `DataLoader`

---

**Next notebook:** [04 - Fine-Tuning with HuggingFace Transformers (Optional)](04_FineTuning_with_HuggingFace_Transformers_optional.ipynb) -- leverage pretrained Transformer models for transfer learning.